# Economy of Bits — Notebook 08: attribution

Notebook 07's out-of-sample test worked, and half-confirmed the hypothesis in the way a real
test should. The **fabricator decline is a thirty-year structural trend** (1996–2015 slope
−0.186 pp/yr, *p* = 0.0015; Intel 5.19% → 0.38% of the top hundred). The **designer rise is
not** (−0.037, *p* = 0.079 before 2016). The paper can date the second but not call it
structural.

But two problems in notebook 07 change numbers the paper would have quoted.

**It used the unrepaired panel.** Meta is absent 2016–2021 and appears to arrive from
nothing in 2022. In the repaired panel Meta is 1.64% of index value in 2016, so "operates
compute" gained *less* than reported.

**And the design layer's +10.24 points is not a layer result — it is Nvidia:**

| | 2016 | 2025 | change |
|---|---|---|---|
| designs compute | 1.27% | 11.50% | +10.24 |
| **without Nvidia** | 0.95% | 4.10% | **+3.15** |
| Nvidia alone | 0.32% | 7.40% | +7.08 |

Nvidia supplies 69%; Nvidia and Broadcom supply 91%. A seven-firm layer where two firms
carry nine-tenths of the movement is a claim about two firms wearing a category label, and a
referee will say so.

This notebook attributes the result firm by firm, tests every layer for dependence on one
member, and settles the boundary cases — starting with Apple, which owns no fab, designs its
own silicon, and contributed more than the entire design layer without Nvidia.

Upload `bits_inputs_v8.zip`. No downloads; about a minute. **This is the last notebook.**

## 1 · Setup

In [ ]:
# =============================================================================
# CELL 1 - Setup
#
# Notebook 07's thirty-year test worked, and it half-confirmed the hypothesis in
# the way a real out-of-sample test should:
#
#   fabricates compute   1996-2015 slope -0.186 pp/yr, p = 0.0015   CONFIRMED
#   designs compute      1996-2015 slope -0.037 pp/yr, p = 0.079    NOT confirmed
#
# The decline of the firms that own fabs is a thirty-year structural trend --
# Intel went from 5.19% of the top hundred by value to 0.38%. The rise of the
# firms that design without fabs is recent, and the paper must date it rather
# than call it structural.
#
# But two problems in notebook 07 have to be fixed before any of this is
# written down, and both change numbers the paper would have quoted.
#
# ONE. It used the ORIGINAL JRFM panel, not the repaired one. Meta is therefore
# absent from 2016-2021 and appears to arrive from nothing in 2022. In the
# repaired panel Meta is 1.64% of index value in 2016, so "operates compute"
# gained LESS than notebook 07 reported: Meta contributes +1.07 points, not
# +2.71.
#
# TWO, and more serious. The design layer's +10.24 points is not a layer result.
# It is Nvidia:
#
#     designs compute, total          1.27%  ->  11.50%   (+10.24)
#     designs compute, without Nvidia 0.95%  ->   4.10%   (+3.15)
#     Nvidia alone                    0.32%  ->   7.40%   (+7.08)
#
# Nvidia supplies 69% of it, and Nvidia plus Broadcom supply 91%. A seven-firm
# layer result where two firms carry nine-tenths of the movement is a claim
# about two firms wearing a category label, and a referee will say so.
#
# This notebook attributes the result firm by firm, tests every layer for
# dependence on a single member, and settles the classification questions the
# rule leaves open -- starting with Apple, which owns no fab, designs its own
# silicon, and is bigger than the entire design layer without Nvidia.
# =============================================================================
import os, re, sys, json, time, glob, zipfile, pathlib, datetime, shutil, warnings
import numpy as np, pandas as pd
import statsmodels.api as sm
from scipy import stats as st

warnings.filterwarnings("ignore")
pd.set_option("display.width", 230); pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 200)

RUN_TAG = "bits_08_attribution"
BASE = pathlib.Path("/content") if pathlib.Path("/content").exists() else pathlib.Path.cwd()
ROOT = BASE / f"economy_of_bits_{RUN_TAG}"
OUTD, FIGS = ROOT / "outputs", ROOT / "outputs" / "figures"
for d in (OUTD, FIGS):
    d.mkdir(parents=True, exist_ok=True)
saved_tables = []

def savetable(df, name):
    if df is None or not len(df):
        print(f"  skipped {name} (empty)"); return
    df.to_parquet(OUTD / f"{name}.parquet"); df.to_csv(OUTD / f"{name}.csv", index=False)
    saved_tables.append(name); print(f"  saved {name}  ({len(df):,} rows)")

def trend(s, maxlags=3):
    s = s.dropna()
    if len(s) < 5:
        return np.nan, np.nan
    X = sm.add_constant(s.index.values.astype(float) - np.mean(s.index.values))
    f = sm.OLS(s.values, X).fit(cov_type="HAC", cov_kwds={"maxlags": maxlags})
    return float(f.params[1]), float(f.pvalues[1])

print(f"writing to: {ROOT}")

## 2 · The repaired panel, firm identity by CIK

In [ ]:
# =============================================================================
# CELL 2 - The repaired panel, and firm identity by CIK
#
# Two fixes applied before anything is computed.
#
# The REPAIRED panel is used for 2016-2025. The original is kept only for the
# thirty-year view, where no repair exists, and that limitation is stated
# wherever the long series appears.
#
# Firm identity keys on CIK. In notebook 07 Alphabet showed as GOOGL +6.18
# points and GOOG -2.90 because the panel keeps whichever share class was larger
# that year, so the firm appears to gain under one ticker and collapse under the
# other. Its actual contribution is the sum, +3.28.
# =============================================================================
def find_inputs():
    for z in sorted(glob.glob(str(BASE / "**" / "bits_inputs_v8.zip"), recursive=True)):
        dest = ROOT / "inputs"; dest.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
        print(f"  extracted {pathlib.Path(z).name}"); return dest
    for d in (BASE / "inputs8", BASE):
        if (pathlib.Path(d) / "panel_repaired.csv.gz").exists():
            return pathlib.Path(d)
    return None

SRC = find_inputs()
if SRC is None:
    raise FileNotFoundError("bits_inputs_v8.zip not found.")

rep = pd.read_csv(SRC / "panel_repaired.csv.gz"); rep["year"] = rep.year.astype(int)
full = pd.read_csv(SRC / "panel_full_1996.csv.gz"); full["year"] = full.year.astype(int)
tick2cik = pd.read_csv(SRC / "ticker_to_cik.csv")
long30 = pd.read_csv(SRC / "layer_share_top100_30y.csv")
CIK = dict(zip(tick2cik.ticker, tick2cik.cik))
CIK.update({"XOM": 34088, "BLK": 1364742})

for df in (rep, full):
    df["cik"] = df.ticker.map(CIK)
    df["firm_id"] = df.cik.fillna(-1).astype(int).astype(str)
    df.loc[df.cik.isna(), "firm_id"] = df.loc[df.cik.isna(), "ticker"]
nm = rep.sort_values("mktcap", ascending=False).groupby("firm_id").ticker.first()
rep["firm"] = rep.firm_id.map(nm)
full["firm"] = full.firm_id.map(nm).fillna(full.ticker)

est = rep[rep.year.between(2016, 2025)].copy()
# One row per firm-year: Alphabet's two classes are one company.
est = (est.groupby(["year", "firm_id", "firm"], as_index=False)
       .agg(mktcap=("mktcap", "sum"), sector=("sector", "first")))
est["cap_share"] = est.mktcap / est.groupby("year").mktcap.transform("sum")
print(f"repaired estimation panel: {len(est):,} firm-years, {est.firm_id.nunique()} firms")

chk = est[est.firm.isin(["META", "GOOGL", "NVDA", "AAPL"])]
print("\nsanity check, % of index value:")
for f_ in ["META", "GOOGL", "NVDA", "AAPL"]:
    s = 100 * chk[chk.firm == f_].set_index("year").cap_share
    if len(s):
        print(f"  {f_:<6} {s.iloc[0]:5.2f}% ({s.index[0]}) -> {s.iloc[-1]:5.2f}% ({s.index[-1]})")
print("\nMeta at ~1.6% in 2016 rather than absent confirms the repaired panel is")
print("in use. Alphabet as one firm confirms the CIK key is working.")

## 3 · Who actually gained the value

In [ ]:
# =============================================================================
# CELL 3 - Who actually gained the value
#
# The decomposition is exact arithmetic, not estimation: a firm's contribution
# is its change in share of index value, and the contributions sum to zero
# because shares of a whole must. The only question is how CONCENTRATED the
# contributions are, and that is what decides whether the paper can speak about
# layers or has to speak about firms.
# =============================================================================
f16 = est[est.year == 2016].set_index("firm_id")
f25 = est[est.year == 2025].set_index("firm_id")
ids = sorted(set(f16.index) | set(f25.index))
contrib = pd.DataFrame({
    "firm": [f25.firm.get(i, f16.firm.get(i, i)) for i in ids],
    "share_2016": [100 * f16.cap_share.get(i, 0.0) for i in ids],
    "share_2025": [100 * f25.cap_share.get(i, 0.0) for i in ids]}, index=ids)
contrib["contribution_pp"] = (contrib.share_2025 - contrib.share_2016).round(3)
contrib = contrib.sort_values("contribution_pp", ascending=False)

print("the twenty firms that gained the most index share, 2016-2025:")
display(contrib.head(20).round(2))
print("\nthe ten that lost the most:")
display(contrib.tail(10).round(2))
savetable(contrib.reset_index().rename(columns={"index": "firm_id"}),
          "firm_contributions")

gain = contrib[contrib.contribution_pp > 0].contribution_pp
print(f"\ntotal gained by firms that gained: {gain.sum():+.2f} points across "
      f"{len(gain)} firms")
print(f"  the single largest gainer is {gain.index[0]} "
      f"({contrib.firm.iloc[0]}) at {gain.iloc[0]:+.2f}, which is "
      f"{100*gain.iloc[0]/gain.sum():.0f}% of all gains")
for k in (1, 2, 3, 5, 10):
    print(f"  top {k:>2} gainers: {gain.head(k).sum():6.2f} points "
          f"({100*gain.head(k).sum()/gain.sum():4.0f}% of all gains)")
hhi = float(((gain / gain.sum()) ** 2).sum())
print(f"\nHerfindahl of the gains: {hhi:.3f}  (1/{1/hhi:.1f} effective firms)")
print("This number governs how the paper may speak. A concentration result whose")
print("own gains are concentrated in a handful of firms is a statement about")
print("those firms; the category is a description of them, not an explanation.")
savetable(pd.DataFrame([{"metric": "total_gains_pp", "value": round(float(gain.sum()), 3)},
                        {"metric": "top1_share_of_gains", "value": round(float(gain.iloc[0]/gain.sum()), 3)},
                        {"metric": "top5_share_of_gains", "value": round(float(gain.head(5).sum()/gain.sum()), 3)},
                        {"metric": "hhi_of_gains", "value": round(hhi, 4)},
                        {"metric": "effective_firms", "value": round(1/hhi, 1)}]),
          "gain_concentration")

## 4 · The boundary cases — Apple and Tesla

In [ ]:
# =============================================================================
# CELL 4 - The classification questions the rule leaves open
#
# "Who owns the fab" is a clean rule but it does not answer everything, and two
# of the largest firms sit exactly on the boundary.
#
# APPLE owns no fab and designs its own silicon -- the M and A series are
# Apple's designs, fabricated by TSMC. Under the rule as written Apple is a
# designer. But Apple's revenue is overwhelmingly assembled hardware, so calling
# it a chip designer would put the world's second-largest firm into a category
# on the strength of a component it does not sell. It contributed +3.26 points,
# more than the entire design layer without Nvidia.
#
# TESLA owns no fab, designs its own inference silicon, and contributed +2.44.
# The same argument applies with less force.
#
# Rather than pick one and hope, the decomposition is reported under all three
# treatments and the paper quotes the range. A result that depends on which side
# of a boundary Apple falls is a result about Apple.
# =============================================================================
DESIGNS = ["NVDA", "AVGO", "QCOM", "MRVL", "XLNX", "ALTR", "BRCM", "LSI",
           "SNPS", "CDNS", "ANET", "AMD"]
FABRICATES = ["INTC", "TXN", "MU", "ADI", "MCHP", "ON", "NXPI", "SWKS", "QRVO",
              "MPWR", "NSM", "LLTC", "MXIM", "FCS", "IDTI", "CY", "ATML"]
EQUIPMENT = ["AMAT", "LRCX", "KLAC", "TER", "NVLS", "KLIC", "ENTG"]
OPERATES = ["MSFT", "GOOGL", "GOOG", "AMZN", "META", "FB", "ORCL"]
DIGITAL_SECTORS = {"Information Technology", "Communication Services"}
DIGITAL_EXTRA = {"AMZN", "TSLA", "NFLX", "BKNG", "EBAY", "ABNB", "UBER", "DASH", "SHOP"}

def layer_of(t, digital, borderline):
    t = str(t).upper()
    if borderline == "designers" and t in ("AAPL", "TSLA"):
        return "designs compute"
    if t == "AAPL" and borderline == "apple_designer":
        return "designs compute"
    if t in DESIGNS:
        return "designs compute"
    if t in FABRICATES:
        return "fabricates compute"
    if t in EQUIPMENT:
        return "compute equipment"
    if t in OPERATES:
        return "operates compute"
    return "other digital" if digital else "not digital"

est["digital"] = (est.sector.isin(DIGITAL_SECTORS)
                  | est.firm.str.upper().isin(DIGITAL_EXTRA))
COMPUTE = ["designs compute", "operates compute", "fabricates compute",
           "compute equipment"]
out = []
for mode, lab in [("baseline", "Apple and Tesla as other digital"),
                  ("apple_designer", "Apple as a designer"),
                  ("designers", "Apple and Tesla as designers")]:
    est["layer"] = [layer_of(t, d, mode) for t, d in zip(est.firm, est.digital)]
    sh = 100 * est.groupby(["year", "layer"]).cap_share.sum().unstack()
    ch = (sh.iloc[-1] - sh.iloc[0]).round(2)
    for L in sh.columns:
        out.append({"treatment": lab, "layer": L,
                    "share_2016": round(float(sh[L].iloc[0]), 2),
                    "share_2025": round(float(sh[L].iloc[-1]), 2),
                    "change_pp": float(ch[L])})
sens = pd.DataFrame(out)
piv = sens.pivot(index="layer", columns="treatment", values="change_pp")
print("change in share of index value, under three treatments of the boundary:")
display(piv.round(2))
savetable(sens, "classification_sensitivity")
print("\nThe operates row should barely move -- no boundary case is in it. If the")
print("designs row swings widely, the paper reports the range and names the")
print("firms that cause it rather than choosing the flattering treatment.")

# Restore the baseline for everything that follows.
est["layer"] = [layer_of(t, d, "baseline") for t, d in zip(est.firm, est.digital)]
base_sh = 100 * est.groupby(["year", "layer"]).cap_share.sum().unstack()
print("\nBASELINE decomposition on the REPAIRED panel:")
dec = pd.DataFrame({"share_2016": base_sh.iloc[0].round(2),
                    "share_2025": base_sh.iloc[-1].round(2)})
dec["change_pp"] = (dec.share_2025 - dec.share_2016).round(2)
display(dec.sort_values("change_pp", ascending=False))
savetable(dec.reset_index(), "decomposition_repaired")
print("\nCompare with notebook 07, which used the unrepaired panel and had")
print("operates compute at 8.53 -> 19.69. Meta's missing early years inflated it.")

## 5 · Leave one out: a category, or one firm?

In [ ]:
# =============================================================================
# CELL 5 - Leave one out: does any layer survive losing its largest member?
#
# This is the test that decides what the paper can claim. For each layer, drop
# its single biggest contributor and recompute. A layer whose result vanishes
# is one firm; a layer whose result merely shrinks is a category.
# =============================================================================
rows = []
for L in COMPUTE + ["other digital"]:
    members = est[est.layer == L]
    if not len(members):
        continue
    c = (contrib[contrib.firm.isin(members.firm.unique())]
         .sort_values("contribution_pp", ascending=False))
    total = float(c.contribution_pp.sum())
    if not len(c):
        continue
    top1, top2 = c.iloc[0], (c.iloc[1] if len(c) > 1 else None)
    rows.append({
        "layer": L, "n_firms": int(members.firm.nunique()),
        "total_pp": round(total, 2),
        "largest": top1.firm, "largest_pp": round(float(top1.contribution_pp), 2),
        "share_from_largest": round(float(top1.contribution_pp) / total, 3) if total else np.nan,
        "ex_largest_pp": round(total - float(top1.contribution_pp), 2),
        "second": (top2.firm if top2 is not None else None),
        "ex_top2_pp": round(total - float(top1.contribution_pp)
                            - (float(top2.contribution_pp) if top2 is not None else 0), 2)})
loo = pd.DataFrame(rows).sort_values("total_pp", ascending=False)
print("leave-one-out by layer:")
display(loo)
savetable(loo, "leave_one_out_by_layer")

print("\nthe design layer, year by year, with and without Nvidia:")
des = est[est.layer == "designs compute"]
tot_des = 100 * des.groupby("year").cap_share.sum()
ex_nvda = 100 * des[des.firm != "NVDA"].groupby("year").cap_share.sum()
ex_two = 100 * des[~des.firm.isin(["NVDA", "AVGO"])].groupby("year").cap_share.sum()
d3 = pd.DataFrame({"designs_total": tot_des, "ex_NVDA": ex_nvda,
                   "ex_NVDA_and_AVGO": ex_two}).round(2)
display(d3)
for c in d3.columns:
    b, p = trend(d3[c])
    print(f"  {c:<20} {d3[c].iloc[0]:5.2f} -> {d3[c].iloc[-1]:5.2f}   "
          f"slope {b:+.4f} pp/yr   p = {p:.4f}")
savetable(d3.reset_index(), "designs_layer_ex_nvidia")

print("\nthe operating layer, same treatment:")
ops = est[est.layer == "operates compute"]
tot_ops = 100 * ops.groupby("year").cap_share.sum()
biggest = (contrib[contrib.firm.isin(ops.firm.unique())]
           .sort_values("contribution_pp", ascending=False).iloc[0].firm)
ex_big = 100 * ops[ops.firm != biggest].groupby("year").cap_share.sum()
o3 = pd.DataFrame({"operates_total": tot_ops, f"ex_{biggest}": ex_big}).round(2)
display(o3)
for c in o3.columns:
    b, p = trend(o3[c])
    print(f"  {c:<20} {o3[c].iloc[0]:5.2f} -> {o3[c].iloc[-1]:5.2f}   "
          f"slope {b:+.4f} pp/yr   p = {p:.4f}")
savetable(o3.reset_index(), "operates_layer_ex_largest")
print("\nIf the operating layer survives dropping its largest member and the")
print("design layer does not survive dropping Nvidia, the paper says exactly")
print("that: a broad reallocation towards the firms that run compute, and a")
print("single firm at the design layer whose rise is the story of one company.")

## 6 · Every number the paper will quote

In [ ]:
# =============================================================================
# CELL 6 - Every number the paper will quote, in one file
#
# Written so the drafting stage never has to re-derive anything or go hunting
# through eight notebooks. Each row carries its source so a co-author or referee
# can trace it.
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
INK, ACC, GREY, GRN = "#1f3b57", "#c8702a", "#7a7a7a", "#4a7c59"
plt.rcParams.update({"font.family": "serif", "font.size": 9, "figure.dpi": 150,
                     "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.alpha": 0.25})

def savefig(fig, name):
    for ext in ("pdf", "png"):
        fig.savefig(FIGS / f"{name}.{ext}", bbox_inches="tight")
    plt.close(fig); print(f"  saved figure {name}")

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
ax = axes[0]
top = contrib.head(12).iloc[::-1]
ax.barh(range(len(top)), top.contribution_pp,
        color=[INK if v > 2 else ACC for v in top.contribution_pp], height=0.65)
ax.set_yticks(range(len(top))); ax.set_yticklabels(top.firm, fontsize=8)
ax.set_xlabel("change in share of index value, 2016-2025 (pp)")
ax.set_title("Twelve firms, and Nvidia")

ax = axes[1]
ax.plot(d3.index, d3.designs_total, "o-", color=INK, lw=2, ms=4, label="designs compute")
ax.plot(d3.index, d3.ex_NVDA, "o--", color=ACC, lw=2, ms=4, label="without Nvidia")
ax.plot(o3.index, o3.operates_total, "s-", color=GRN, lw=2, ms=4, label="operates compute")
ax.set_xlabel("year"); ax.set_ylabel("% of index value")
ax.set_title("One firm, or a category?"); ax.legend(fontsize=8)

ax = axes[2]
L30 = long30.set_index("year")
for c, col, ls in [("designs compute", INK, "-"), ("operates compute", ACC, "-"),
                   ("fabricates compute", GRN, "--")]:
    if c in L30.columns:
        s = L30[c].dropna()
        ax.plot(s.index, s.values, ls, color=col, lw=2, label=c.replace(" compute", ""))
ax.axvspan(2016, 2025, color=INK, alpha=0.06, lw=0)
ax.set_xlabel("year"); ax.set_ylabel("% of top-100 value")
ax.set_title("Thirty years (unrepaired panel)"); ax.legend(fontsize=7.5)
fig.suptitle("Notebook 08: attribution", fontsize=10, color=GREY, y=1.02)
savefig(fig, "A1_attribution")

ineq = pd.read_csv(SRC / "inequality_by_year.csv")
alpha = pd.read_csv(SRC / "pareto_alpha_by_year.csv")
facts = []
def add(section, name, value, source):
    facts.append({"section": section, "quantity": name, "value": value, "source": source})

add("paper 1", "top 1% share 2016 -> 2025 (%)", "11.3 -> 30.0", "JRFM, confirmed nb03")
add("paper 1", "Hill alpha 2016 -> 2025 (repaired)", "1.4669 -> 1.1413, p=0.0062", "nb03")
add("paper 1", "Gini 2016 -> 2025 (repaired)", "0.5879 -> 0.7186", "nb03")
for _, r in dec.reset_index().iterrows():
    add("decomposition", f"{r['layer']} share change (pp)", r.change_pp, "nb08 cell 4")
for _, r in loo.iterrows():
    add("attribution", f"{r.layer}: largest member",
        f"{r.largest} = {r.share_from_largest:.0%} of the layer's gain", "nb08 cell 5")
add("attribution", "Herfindahl of firm gains", round(hhi, 4), "nb08 cell 3")
add("attribution", "effective number of gaining firms", round(1 / hhi, 1), "nb08 cell 3")
add("mechanism", "hyperscaler vs other tail, capex extra trend", "+0.0139, p=0.0003", "nb06")
add("mechanism", "tail vs everyone, capex extra trend", "+0.0038, p=0.27", "nb06")
add("mechanism", "capex predicts share gain", "-0.115, p=0.067", "nb06 bridge")
add("out of sample", "fabricates share slope 1996-2015", "-0.186 pp/yr, p=0.0015", "nb07")
add("out of sample", "designs share slope 1996-2015", "-0.037 pp/yr, p=0.079", "nb07")
add("out of sample", "Intel share of top 100", "5.19% (1996) -> 0.38% (2025)", "nb07")
add("hypotheses", "H1 bit intensity vs tail (sector-neutral)", "+0.299, p=0.0002 WRONG SIGN", "nb05")
add("hypotheses", "H3 Gibrat, growth on size", "+0.007, p=0.12, not rejected", "nb04")
add("hypotheses", "H4 firms in top 1% all ten years", "AAPL, MSFT only", "nb04")
paper_facts = pd.DataFrame(facts)
# The value column deliberately mixes numbers and strings like "1.49 -> 1.14,
# p=0.006", which parquet will not store. Cast to text: this table is for
# reading and quoting, not for arithmetic.
paper_facts["value"] = paper_facts.value.astype(str)
display(paper_facts)
savetable(paper_facts, "paper_facts")

files = sorted(p for p in OUTD.rglob("*") if p.is_file())
pd.DataFrame([{"file": str(p.relative_to(OUTD)),
               "kb": round(p.stat().st_size / 1024, 1)} for p in files]).to_csv(
    OUTD / "MANIFEST.csv", index=False)
(OUTD / "run_summary.json").write_text(json.dumps(
    {"run": RUN_TAG, "when": datetime.datetime.now().isoformat(timespec="seconds"),
     "tables": saved_tables}, indent=2, default=str))

print("\n" + "=" * 70)
print("SEND ME")
print("=" * 70)
print("  firm_contributions.csv + gain_concentration.csv")
print("  leave_one_out_by_layer.csv + designs_layer_ex_nvidia.csv")
print("  classification_sensitivity.csv + decomposition_repaired.csv")
print("  paper_facts.csv")
print("\nThis is the last notebook. Whatever it says, the next thing I send is a")
print("draft -- the claim will be shaped by these numbers rather than waiting")
print("for better ones.")

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
archive = shutil.make_archive(str(BASE / f"economy_of_bits_{RUN_TAG}_{stamp}"),
                              "zip", root_dir=OUTD.parent, base_dir=OUTD.name)
print(f"\nzipped -> {archive}  ({pathlib.Path(archive).stat().st_size/1024:.0f} KB)")
try:
    from google.colab import files as colab_files
    colab_files.download(archive); print("Download started.")
except ImportError:
    print("Not on Colab. Copy the archive from the path above.")